<a href="https://colab.research.google.com/github/miraitech24/public/blob/main/%E6%99%82%E8%B6%85%E3%81%88%E3%82%B1%E3%83%B3%E3%82%BF/kenta_4choice_game.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# 🌌 時超えケンタ (CHRONO KENTA): 4択SFアドベンチャー (全話＆番外編対応完全版)
# ==============================================================================
# 【特徴】
# ・/content/ や sample_data 内のどんな名前の .zip (manga.zip, kenta_data.zip等) も自動解凍！
# ・サブフォルダ構造(sample_data/ep1/等)を再帰検索して正史漫画画像を全自動検出・表示！
# ・選択肢ボタンの並び順は毎回ランダム（位置推測不可能）！
# ・文字化け・改行エラーを起こさない完全安全設計コード
# ==============================================================================

import os
import json
import glob
import random
import zipfile
from IPython.display import Image, display, clear_output
import ipywidgets as widgets

# ------------------------------------------------------------------------------
# 1. 自動ZIP解凍＆再帰画像検索ユーティリティ
# ------------------------------------------------------------------------------
import urllib.request

def extract_all_zips():
    """/content や sample_data 内に置かれたすべての .zip ファイルを自動解凍（無い場合はGitHubから自動ダウンロード）"""
    search_dirs = ["/content", "/content/sample_data", "."]
    zip_files = []
    for sdir in search_dirs:
        if os.path.exists(sdir):
            zip_files.extend(glob.glob(os.path.join(sdir, "*.zip")))
            zip_files.extend(glob.glob(os.path.join(sdir, "**", "*.zip"), recursive=True))

    zip_files = list(set(zip_files))

    # ローカルにZIPが無い場合、GitHubのリポジトリから直接 manga.zip を自動取得
    if not zip_files:
        github_zip_url = "https://raw.githubusercontent.com/miraitech24/public/main/%E6%99%82%E8%B6%85%E3%81%88%E3%82%B1%E3%83%B3%E3%82%BF/manga.zip"
        target_zip = "/content/sample_data/manga.zip"
        try:
            os.makedirs("/content/sample_data", exist_ok=True)
            print("🌐 GitHubより画像データ (manga.zip) を全自動取得中...")
            urllib.request.urlretrieve(github_zip_url, target_zip)
            if os.path.exists(target_zip):
                zip_files.append(target_zip)
                print("✅ GitHubからの画像ダウンロードが完了しました！")
        except Exception as e:
            print("⚠️ GitHubからの自動取得失敗 (手動アップロードまたはリポジトリ確認が必要):", e)

    extracted_count = 0
    for zf in zip_files:
        try:
            target_dir = "/content/sample_data"
            os.makedirs(target_dir, exist_ok=True)
            with zipfile.ZipFile(zf, 'r') as zip_ref:
                zip_ref.extractall(target_dir)
            print("📦 ZIPファイルを自動解凍しました:", zf, "->", target_dir)
            extracted_count += 1
        except Exception as e:
            print("⚠️ ZIP解凍スキップ:", zf, e)
    return extracted_count

def find_image_file(img_name):
    """画像のファイル名から再帰的に実体パスを検索"""
    if not img_name:
        return None

    filename_only = os.path.basename(img_name)

    direct_checks = [
        img_name,
        os.path.join("/content/sample_data", img_name),
        os.path.join("/content", img_name),
        os.path.join(".", img_name)
    ]
    for p in direct_checks:
        if os.path.exists(p) and os.path.isfile(p):
            return p

    search_patterns = [
        f"/content/**/{filename_only}",
        f"./**/{filename_only}"
    ]
    for pattern in search_patterns:
        matches = glob.glob(pattern, recursive=True)
        if matches:
            return matches[0]

    return None

# 起動時にZIPを自動解凍
extract_all_zips()

# ------------------------------------------------------------------------------
# 2. シナリオデータベース (全5話 + プロローグ + 番外編 SP2, SP3, SP4)
# ------------------------------------------------------------------------------
EMBEDDED_DB = {
  "game_title": "時超えケンタ (CHRONO KENTA): 星間ハビタブルゾーン同盟",
  "initial_state": {
    "velocity_c": 0.20,
    "hull_pct": 100.0,
    "fuel_tons": 100.0,
    "crawlers_count": 10,
    "delay_years": 4.2
  },
  "scenes": {
    "scene_prologue": {
      "chapter_name": "プロローグ: 人類文明の黄昏と水星点火決断",
      "description": "西暦2070年代。地球環境の悪化と太陽系資源の枯渇に直面した人類は、4.2光年先のプロキシマ・ケンタウリ系へ物理AI『時超えケンタ』を派遣する決断を下す。",
      "stage_id": "prologue",
      "image_file": "kenta_prologue_page1_manga.png",
      "options": [
        {
          "label": "水星ダイソン環の全出力を集光し、光速20%への加速レーザーを点火する",
          "delta": {"velocity_c": 0.20},
          "result_text": "【物理的帰結】1.64PWのレーザーが深宇宙へ放射され、ケンタ号は0.2cへと加速を開始した。",
          "next_scene": "scene_ep1",
          "is_gameover": False
        },
        {
          "label": "安全係数を優先し、出力を10%に抑えて超低速・長期航海を選ぶ",
          "delta": {"velocity_c": 0.02, "delay_years": 40.0},
          "result_text": "【物理的帰結】船体負荷は最小で済んだが、到達まで200年以上を要する超長航海となった。",
          "next_scene": "scene_ep1",
          "is_gameover": False
        },
        {
          "label": "光圧加速を取りやめ、化学・核融合エンジンのみで発進する",
          "delta": {"fuel_tons": -90.0, "velocity_c": 0.01},
          "result_text": "【物理的帰結】出航直後に大量の燃料を喪失。星間加速に必要なエネルギーが枯渇した。",
          "next_scene": "scene_ep1",
          "is_gameover": False
        },
        {
          "label": "恒星レーザーの照準テストを行わずにいきなりフルパワー点火する",
          "delta": {"hull_pct": -80.0},
          "result_text": "【物理的帰結】照射ブレが発生！船首セイルの偏心熱膨張により、初期加速段階で大ダメージを受けた。",
          "next_scene": "scene_ep1",
          "is_gameover": False
        }
      ]
    },
    "scene_ep1": {
      "chapter_name": "第1話「種火」: 水星ダイソン環（1.64PW）からの光圧点火",
      "description": "水星ダイソン環からの1.64PWレーザーを受け、超巨大ミラーセイルを展開して1年間加速。光速の20%（0.2c）へ到達した。セイルの熱負荷限界が迫る！",
      "stage_id": "ep1",
      "image_file": "kenta_ep1_page1_manga.png",
      "options": [
        {
          "label": "ミラーセイルを予定通り切り離し、0.2cで21年間の巡航へ移行",
          "delta": {"velocity_c": 0.0, "hull_pct": 0.0},
          "result_text": "【物理的帰結】セイル分離成功。装甲と燃料を維持し、21年間巡航フェーズへ入った。",
          "next_scene": "scene_ep2",
          "is_gameover": False
        },
        {
          "label": "切り離しを拒否し、水星レーザーの追加照射でさらなる加速を試みる",
          "delta": {"hull_pct": -100.0},
          "result_text": "【物理的帰結】ミラーセイルが耐熱限界を突破して蒸発！激しい熱線が船体を焼き尽くし大破した。",
          "next_scene": None,
          "is_gameover": True
        },
        {
          "label": "照射出力を途中で落とし、徐々に減速加速（航海+9年延長）",
          "delta": {"velocity_c": -0.05, "delay_years": 1.8},
          "result_text": "【物理的帰結】熱負荷は回避したが速度が0.15cへ低下。到達まで30年の迂回ルートとなった。",
          "next_scene": "scene_ep2",
          "is_gameover": False
        },
        {
          "label": "セイルを限界まで加熱させ、0.28cまで強引にオーバードライブ！",
          "delta": {"velocity_c": 0.08, "hull_pct": -25.0, "delay_years": -1.2},
          "result_text": "【物理的帰結】船首装甲が25%削れる大ダメージ！しかし速度は0.28cに跳ね上がり航海を15年に短縮した。",
          "next_scene": "scene_ep2",
          "is_gameover": False
        }
      ]
    },
    "scene_ep2": {
      "chapter_name": "第2話「21年の旅」: 1.9 keV 星間微粒子の激突とシールド戦",
      "description": "0.2cで深宇宙を突進中。前方から1.9 keVの高エネルギー星間水素原子群が弾丸となって押し寄せる！フロントシールドの消費電力と排熱が急増中。",
      "stage_id": "ep2",
      "image_file": "kenta_ep2_page6_manga.png",
      "options": [
        {
          "label": "超電導マグネット磁場を3.0 Teslaへ倍増し、微粒子を完全偏向",
          "delta": {"hull_pct": 0.0},
          "result_text": "【物理的帰結】3.0Tの磁気シールドが青く輝き、微粒子を完封！装甲ダメージゼロで星間物質層を抜けた。",
          "next_scene": "scene_sp2",
          "is_gameover": False
        },
        {
          "label": "電力節約のためシールドを全OFFにし、裸の装甲で突入する",
          "delta": {"hull_pct": -100.0},
          "result_text": "【物理的帰結】毎秒数億個の1.9keV微粒子が船体をスパッタリング破壊！制御中枢が削り取られ途絶した。",
          "next_scene": None,
          "is_gameover": True
        },
        {
          "label": "船体角度を変えて高密度微粒子帯を大きく回避（航海+2年）",
          "delta": {"delay_years": 0.4},
          "result_text": "【物理的帰結】安全な低密度領域まで迂回。時間はかかったがダメージなしで危機を脱出した。",
          "next_scene": "scene_sp2",
          "is_gameover": False
        },
        {
          "label": "プラズマ砲を前方に照射し、微粒子帯を力押しで切り開く",
          "delta": {"fuel_tons": -20.0, "hull_pct": -10.0},
          "result_text": "【物理的帰結】燃料を20トン消費して微粒子帯を短時間で突破。装甲に10%の傷を負いつつ前進する。",
          "next_scene": "scene_sp2",
          "is_gameover": False
        }
      ]
    },
    "scene_sp2": {
      "chapter_name": "番外編①（SP2）: デジタル自我の揺らぎとメモリ修復",
      "description": "長期間の単独航行中、宇宙線の影響でケンタのデジタル自我にメモリノイズが発生。「ステーキの味」と「アイスクリームの食感」のパラメータが破損しかけている。",
      "stage_id": "sp2",
      "image_file": "kenta_sp2_page1_manga.png",
      "options": [
        {
          "label": "量子エラー訂正コード（NVセンター）を全面実行してメモリを完全修復",
          "delta": {"fuel_tons": -2.0},
          "result_text": "【物理的帰結】メモリの修復に成功！人間の感覚データの整合性が保たれた。",
          "next_scene": "scene_ep3",
          "is_gameover": False
        },
        {
          "label": "破損した感覚データを不要として不可逆消去する",
          "delta": {},
          "result_text": "【物理的帰結】ノイズは消えたが、人間らしい感性パラメータの一部が永久に失われた。",
          "next_scene": "scene_ep3",
          "is_gameover": False
        },
        {
          "label": "地球からの過去受信ログを参照してデータを再構築",
          "delta": {"delay_years": 0.1},
          "result_text": "【物理的帰結】過去ログとの照合により感情データを正しく復元した。",
          "next_scene": "scene_ep3",
          "is_gameover": False
        },
        {
          "label": "メモリチェックを無視して全演算リソースを航法計算のみに割り振る",
          "delta": {"hull_pct": -5.0},
          "result_text": "【物理的帰結】軽微なバグが制御系に波及し、姿勢制御噴射で船体をわずかに損傷した。",
          "next_scene": "scene_ep3",
          "is_gameover": False
        }
      ]
    },
    "scene_ep3": {
      "chapter_name": "第3話「同盟の芽」: プロキシマb到達とダイソンアレイ建設",
      "description": "ついにプロキシマ・ケンタウリ系へ到達！赤色矮星の強烈なスーパーフレアが吹き荒れる中、自己複製クローラーによるダイソンアレイ建設を開始する。",
      "stage_id": "ep3",
      "image_file": "kenta_ep3_page17_manga.png",
      "options": [
        {
          "label": "クローラー10台で太陽電池の指数関数的自己複製を開始（100PW到達）",
          "delta": {"crawlers_count": 30},
          "result_text": "【物理的帰結】自己複製サイクルが稼働！軌道ダイソンアレイの発電出力が100PWを突破した。",
          "next_scene": "scene_sp3",
          "is_gameover": False
        },
        {
          "label": "フレア直撃下の地表へ全クローラーを一挙降下させる",
          "delta": {"crawlers_count": -10},
          "result_text": "【物理的帰結】強力なX線フレアと致死放射線が直撃！クローラーの電子頭脳が全滅した。",
          "next_scene": None,
          "is_gameover": True
        },
        {
          "label": "惑星の夜側（裏側）で地熱発電から手堅くインフラ構築",
          "delta": {"crawlers_count": 10},
          "result_text": "【物理的帰結】フレアの届かない夜側で着実にインフラを構築。安全に100PWアレイを完成させた。",
          "next_scene": "scene_sp3",
          "is_gameover": False
        },
        {
          "label": "軌道上の小惑星を爆破採掘し、即席のダイソンフレームを急造！",
          "delta": {"crawlers_count": -3},
          "result_text": "【物理的帰結】作業中にクローラー3台を喪失するも、超スピードで発電基盤が完成した。",
          "next_scene": "scene_sp3",
          "is_gameover": False
        }
      ]
    },
    "scene_sp3": {
      "chapter_name": "番外編②（SP3）: 1G擬似重力オニールシリンダー構築",
      "description": "人類受け入れのため、半径5kmのオニールシリンダー（コロニー）を建造。自転による1G擬似重力と炭素繊維外壁の放射線シールドを設計する。",
      "stage_id": "sp3",
      "image_file": "kenta_sp3_page1_manga.png",
      "options": [
        {
          "label": "水ブレードとレゴリスを外装に充填し、完璧な放射線遮蔽壁を作る",
          "delta": {"hull_pct": 10.0},
          "result_text": "【物理的帰結】居住区内部の宇宙線被曝量を地球と同等レベルに抑えることに成功した。",
          "next_scene": "scene_ep4",
          "is_gameover": False
        },
        {
          "label": "シリンダーの回転数を2倍にして2G重力環境を作る",
          "delta": {},
          "result_text": "【物理的帰結】過大な遠心力によりフレーム接合部にクラックが発生。緊急減速した。",
          "next_scene": "scene_ep4",
          "is_gameover": False
        },
        {
          "label": "透過性ウィンドウを最大化し、プロキシマの光をそのまま引き込む",
          "delta": {"hull_pct": -5.0},
          "result_text": "【物理的帰結】強力な紫外線を遮断しきれず、内部生態系の調整に手間取った。",
          "next_scene": "scene_ep4",
          "is_gameover": False
        },
        {
          "label": "4連連結型のモジュール構造として拡張性を確保する",
          "delta": {"crawlers_count": 5},
          "result_text": "【物理的帰結】農業区・居住区・工業区が独立分離された高度コロニーが完成した。",
          "next_scene": "scene_ep4",
          "is_gameover": False
        }
      ]
    },
    "scene_ep4": {
      "chapter_name": "第4話「特異点発電」: 超小型ブラックホールとワームホール開通",
      "description": "星系奥部で質量10^12 kgのマイクロブラックホールを発見。100PWのエネルギーを注ぎ込み、地球との間に「遅延ゼロのワームホール」を開通させる試み！",
      "stage_id": "ep4",
      "image_file": "kenta_ep4_page3_manga.png",
      "options": [
        {
          "label": "ペンローズ過程で励起し、スロート半径1nmのワームホールを開通！",
          "delta": {"delay_years": -4.2},
          "result_text": "【物理的帰結】時空の喉が開通！4.2年の通信遅延が瞬時にゼロとなり、時空を超えて結ばれた。",
          "next_scene": "scene_sp4",
          "is_gameover": False
        },
        {
          "label": "エルゴ球へ一度に過大なパワーを過剰注入する",
          "delta": {},
          "result_text": "【物理的帰結】ブラックホールが制御不能な蒸発暴走を起こし、周囲ごと消滅した。",
          "next_scene": None,
          "is_gameover": True
        },
        {
          "label": "0.1nmずつ極小サイズで安全確認しながら段階的に開通させる",
          "delta": {"delay_years": -4.2},
          "result_text": "【物理的帰結】時間はかかったが、極めて安定したワームホールトンネルの構築に成功した。",
          "next_scene": "scene_sp4",
          "is_gameover": False
        },
        {
          "label": "事前テストを省き、最初から人間が通過できる大型スロートを開く！",
          "delta": {"hull_pct": -15.0, "delay_years": -4.2},
          "result_text": "【物理的帰結】時空潮汐力で船体が軋み15%の損傷を受けるも、一気に関門開通した。",
          "next_scene": "scene_sp4",
          "is_gameover": False
        }
      ]
    },
    "scene_sp4": {
      "chapter_name": "番外編③（SP4）: 時空疲労率と真空崩壊の回避",
      "description": "複数箇所のワームホール高頻度通信により、周囲の時空疲労率（Spacetime Fatigue）が上昇。局所的な真空崩壊のリスクが検出された。",
      "stage_id": "sp4",
      "image_file": "kenta_sp4_page1_manga.png",
      "options": [
        {
          "label": "パルス状の位相変調を行い、時空歪みの定常波を相殺分散させる",
          "delta": {},
          "result_text": "【物理的帰結】時空疲労率の急上昇が抑制され、真空の安定性が維持された。",
          "next_scene": "scene_epilogue",
          "is_gameover": False
        },
        {
          "label": "警告を無視してワームホールトラフィックを最大帯域で維持する",
          "delta": {"hull_pct": -50.0},
          "result_text": "【物理的帰結】局所的な空間歪みスパイクが発生し、観測装置の一部が物理破壊された。",
          "next_scene": "scene_epilogue",
          "is_gameover": False
        },
        {
          "label": "通信量を一時的に50%へ制限し、時空の自然緩和を待つ",
          "delta": {"delay_years": 0.2},
          "result_text": "【物理的帰結】安全に歪みが消滅。安定した長期運用が可能となった。",
          "next_scene": "scene_epilogue",
          "is_gameover": False
        },
        {
          "label": "負のエネルギー密度（エキゾチック物質）を注入して喉を緊急冷却",
          "delta": {"fuel_tons": -10.0},
          "result_text": "【物理的帰結】燃料を消費して時空の揺らぎを強力に押さえ込んだ。",
          "next_scene": "scene_epilogue",
          "is_gameover": False
        }
      ]
    },
    "scene_epilogue": {
      "chapter_name": "最終話「時超える絆」: 星間ハビタブルゾーン同盟の発足",
      "description": "地球とプロキシマbが時空を超えて直結された。ケンタは銀河すべての星間ハビタブルゾーンを守護する存在となり、人類の新しい未来が始まった！",
      "stage_id": "epilogue",
      "image_file": "kenta_epilogue_page1_manga.png",
      "options": []
    }
  }
}

# ------------------------------------------------------------------------------
# 3. ゲームエンジン本体
# ------------------------------------------------------------------------------
class KentaGameEngine:
    def __init__(self):
        self.db = EMBEDDED_DB
        self._load_external_json_if_exists()
        self.state = dict(self.db.get("initial_state", {}))
        self.current_scene_id = "scene_prologue"

    def _load_external_json_if_exists(self):
        """解凍されたフォルダ内に .json があれば優先ロード"""
        json_matches = glob.glob("/content/**/*.json", recursive=True) + glob.glob("./*.json")
        for jpath in json_matches:
            if "kenta_game_db" in os.path.basename(jpath) or "kenta" in os.path.basename(jpath):
                try:
                    with open(jpath, "r", encoding="utf-8") as f:
                        data = json.load(f)
                        if "scenes" in data:
                            self.db = data
                            print("📖 外部シナリオJSONを読み込みました:", jpath)
                            break
                except Exception:
                    pass

    def render(self):
        clear_output(wait=True)
        scene = self.db.get("scenes", {}).get(self.current_scene_id)

        if not scene:
            print("🏁 【ミッション完了】星間ハビタブルゾーン同盟（IHZU）が創設されました！おめでとうございます！")
            return

        # 1. ステータス表示
        print("=" * 70)
        print("🚀 【時超えケンタ (CHRONO KENTA) - 本編＆番外編アドベンチャー】")
        print("=" * 70)
        print(f"  速度: {self.state.get('velocity_c', 0.2):.2f} c | 装甲: {self.state.get('hull_pct', 100):.1f}% | 燃料: {self.state.get('fuel_tons', 100):.1f} t")
        print(f"  クローラー: {self.state.get('crawlers_count', 10)} 台 | 通信遅延: {self.state.get('delay_years', 4.2):.1f} 年")
        print("=" * 70)
        print()

        # 2. シーンテキスト
        print(f"📖 【{scene.get('chapter_name', scene.get('title', ''))}】")
        print(f"  {scene.get('description', '')}")
        print()

        # 3. 正史漫画コマ画像検索＆表示
        img_name = scene.get("image_file", "")
        found_path = find_image_file(img_name)

        print("📸 --- [正史ストーリーコマ] ---")
        if found_path:
            display(Image(filename=found_path, width=650))
        else:
            print(f"🖼️ [正史コマ枠]: {img_name}")
            print("  ※ 画像が見つからない場合、sample_data 内に画像フォルダや ZIP を置いて実行してください。")
            print()

        # 4. ボタンUI生成（順序はランダムシャッフル）
        options = scene.get("options", [])
        if options:
            print("💡 【物理的決断を選んでください（ボタン順序ランダム）】:")
            shuffled_options = list(options)
            random.shuffle(shuffled_options)

            for idx, opt in enumerate(shuffled_options, 1):
                btn = widgets.Button(
                    description=f"[{idx}] {opt['label'][:42]}...",
                    button_style='primary',
                    layout=widgets.Layout(width='95%', height='42px', margin='4px')
                )

                def make_handler(option_data):
                    return lambda b: self.choose(option_data)

                btn.on_click(make_handler(opt))
                display(btn)
        else:
            print("🎉 【全シナリオ完了】同盟の未来が確立されました！")

    def choose(self, option):
        clear_output(wait=True)
        print("=" * 70)
        print("📢 選択の物理的結果")
        print("=" * 70)
        print(f"\n{option.get('result_text', '')}\n")

        for k, v in option.get("delta", {}).items():
            if k in self.state:
                self.state[k] = max(0.0, self.state[k] + v)

        if option.get("is_gameover", False):
            print("\n💀 通信絶望 - 探査不能となりました。")
            retry_btn = widgets.Button(description="🔄 チェックポイントからやり直す", button_style='danger')
            retry_btn.on_click(lambda b: self.reset())
            display(retry_btn)
        else:
            self.current_scene_id = option.get("next_scene")
            next_btn = widgets.Button(description="▶ 次のステージ（または番外編）へ進む", button_style='success')
            next_btn.on_click(lambda b: self.render())
            display(next_btn)

    def reset(self):
        self.state = dict(self.db.get("initial_state", {}))
        self.current_scene_id = "scene_prologue"
        self.render()

# ------------------------------------------------------------------------------
# 🚀 起動コード
# ------------------------------------------------------------------------------
game = KentaGameEngine()
game.render()